# WoW Economy Forecaster -- Exploratory Data Analysis

## Introduction

World of Warcraft's Auction House is a real economy with tens of thousands of tradeable items, millions of daily transactions, and price movements driven by a mix of supply/demand fundamentals and game-specific catalysts (patch cycles, seasonal events, competitive raid races).

The **WoW Economy Forecaster** is a local-first ML system that ingests hourly commodity snapshots from the Blizzard API, engineers 48 features across 10 groups, and produces 1-day / 7-day / 28-day price forecasts for every economic archetype on a realm. Forecasts feed a recommendation engine that scores opportunities on a weighted composite of opportunity, liquidity, volatility, event impact, and uncertainty.

This notebook walks through the raw data: what it looks like, how prices distribute across item categories, what temporal patterns exist, and how in-game events create measurable price shocks. The goal is to build intuition before diving into model development (Notebook 02) and evaluation (Notebook 03).

### Why do WoW prices fluctuate?

Since patch 9.2.7, the commodity Auction House is **region-wide** -- every player on US (or EU) realms shares a single order book. This means:

- **Supply** is driven by profession output, farm routes, and world-drop rates across the entire region.
- **Demand** spikes around content releases (new raids, Mythic+ seasons, PvP seasons) when players need consumables, gems, and enchants.
- **Events** like the Race to World First (RTWF) or mid-season patches create predictable demand shocks for specific item categories.
- **Day-of-week effects** emerge because weekend play sessions differ from weekday patterns.

These patterns are the foundation of the feature engineering and forecasting pipeline.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
# Adjust these paths if your environment differs.

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # assumes notebook is in notebooks/
DB_PATH = PROJECT_ROOT / "wow_forecaster.db"
REALM = "us"

print(f"Project root : {PROJECT_ROOT}")
print(f"Database     : {DB_PATH}")
print(f"DB exists    : {DB_PATH.exists()}")

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────

import sqlite3
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from wow_forecaster.viz.theme import (
    CATEGORY_COLORS,
    WOW_PALETTE,
    apply_wow_theme,
)
from wow_forecaster.viz.data_queries import (
    fetch_archetypes,
    fetch_events,
    fetch_forecast_data,
    fetch_historical_prices,
)

warnings.filterwarnings("ignore", category=FutureWarning)

# Apply the WoW dark theme globally
fig_tmp, _ = plt.subplots()
apply_wow_theme(fig_tmp)
plt.close(fig_tmp)

print("Imports OK.")

---
## 1. Data Overview

The forecaster stores all data in a single SQLite database. Key tables include:

| Table | Purpose |
|-------|---------|
| `market_observations_raw` | Raw Blizzard API snapshots (hourly) |
| `market_observations_normalized` | Z-score normalized, outlier-flagged observations |
| `items` | Item metadata with archetype assignment |
| `economic_archetypes` | Category/sub-tag taxonomy for price grouping |
| `wow_events` | In-game events with dates, severity, and announced_at |
| `forecast_outputs` | Model predictions with CI bands |
| `recommendation_outputs` | Scored buy/sell/hold/avoid recommendations |

Let's see what we have.

In [ ]:
# ── Table row counts and date ranges ────────────────────────────────────────

def safe_query(db_path: Path, sql: str, params: list | None = None) -> pd.DataFrame:
    """Execute a SQL query, returning an empty DataFrame on failure."""
    if not db_path.exists():
        print(f"Database not found at {db_path}. Run the pipeline first.")
        return pd.DataFrame()
    try:
        conn = sqlite3.connect(str(db_path))
        df = pd.read_sql_query(sql, conn, params=params or [])
        conn.close()
        return df
    except Exception as exc:
        print(f"Query failed: {exc}")
        return pd.DataFrame()

# Table summary
tables_sql = """
    SELECT name FROM sqlite_master
    WHERE type='table' AND name NOT LIKE 'sqlite_%'
    ORDER BY name
"""
tables = safe_query(DB_PATH, tables_sql)

if not tables.empty:
    summary_rows = []
    conn = sqlite3.connect(str(DB_PATH))
    for tbl in tables["name"]:
        count = pd.read_sql_query(f"SELECT COUNT(*) AS n FROM [{tbl}]", conn).iloc[0]["n"]
        summary_rows.append({"table": tbl, "rows": count})
    conn.close()
    df_summary = pd.DataFrame(summary_rows)
    print(f"Database contains {len(df_summary)} tables:\n")
    print(df_summary.to_string(index=False))
else:
    print("No tables found. Run `wow-forecaster run-hourly-refresh` to populate the DB.")

In [ ]:
# ── Observation date range and coverage ──────────────────────────────────────

coverage_sql = """
    SELECT MIN(date(observed_at)) AS first_date,
           MAX(date(observed_at)) AS last_date,
           COUNT(DISTINCT date(observed_at)) AS distinct_days,
           COUNT(*)                          AS total_rows
    FROM   market_observations_normalized
    WHERE  realm_slug = ?
"""
df_cov = safe_query(DB_PATH, coverage_sql, [REALM])
if not df_cov.empty and df_cov.iloc[0]["total_rows"] > 0:
    row = df_cov.iloc[0]
    print(f"Realm: {REALM}")
    print(f"Date range     : {row['first_date']}  ->  {row['last_date']}")
    print(f"Distinct days  : {row['distinct_days']}")
    print(f"Total obs rows : {row['total_rows']:,}")
else:
    print("No normalized observations found. Run the ingestion pipeline first.")

---
## 2. Price Distributions by Category

The forecaster groups items into **economic archetypes** -- clusters of items that behave similarly in the AH. Each archetype belongs to one of 10 top-level categories:

- **consumable** -- flasks, potions, food (demand-driven by raiding)
- **mat** -- raw materials (herbs, ore, leather)
- **gear** -- crafted and dropped equipment
- **enchant** -- weapon/armor enchantments
- **gem** -- socketed gems
- **prof_tool** -- profession tools
- **reagent** -- profession reagents
- **trade_good** -- general trade goods
- **service** -- profession services
- **collection** -- pets, mounts, transmog

These categories have fundamentally different price ranges and volatility profiles. Let's look at their distributions.

In [ ]:
# ── Fetch archetype metadata ─────────────────────────────────────────────────

df_archetypes = fetch_archetypes(DB_PATH)
if df_archetypes.empty:
    print("No archetype data. Run build-datasets to populate economic_archetypes.")
else:
    print(f"Total archetypes: {len(df_archetypes)}")
    print(f"\nCategory breakdown:")
    print(df_archetypes["category_tag"].value_counts().to_string())

In [ ]:
# ── Price distribution by category ───────────────────────────────────────────
# Fetch recent daily prices across all archetypes to build histograms.

price_dist_sql = """
    SELECT ea.category_tag,
           AVG(n.price_gold) AS avg_price_gold
    FROM   market_observations_normalized n
    JOIN   items i ON n.item_id = i.item_id
    JOIN   economic_archetypes ea ON i.archetype_id = ea.archetype_id
    WHERE  n.realm_slug = ?
      AND  n.is_outlier = 0
      AND  date(n.observed_at) >= date('now', '-30 days')
    GROUP BY ea.category_tag, ea.archetype_id, date(n.observed_at)
    ORDER BY ea.category_tag
"""
df_prices = safe_query(DB_PATH, price_dist_sql, [REALM])

if not df_prices.empty:
    categories = sorted(df_prices["category_tag"].dropna().unique())
    n_cats = len(categories)
    ncols = min(3, n_cats)
    nrows = (n_cats + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
    apply_wow_theme(fig)
    axes_flat = np.array(axes).flatten() if n_cats > 1 else [axes]

    for idx, cat in enumerate(categories):
        ax = axes_flat[idx]
        subset = df_prices[df_prices["category_tag"] == cat]["avg_price_gold"].dropna()
        color = CATEGORY_COLORS.get(cat, WOW_PALETTE["primary"])

        if not subset.empty:
            # Log scale for readability across wide price ranges
            log_prices = np.log10(subset.clip(lower=0.01))
            ax.hist(log_prices, bins=30, color=color, alpha=0.75,
                    edgecolor=WOW_PALETTE["grid"], linewidth=0.5)

        ax.set_title(cat, fontsize=11, fontweight="bold", color=color)
        ax.set_xlabel("log10(Price in gold)")
        ax.set_ylabel("Count")

    # Hide unused axes
    for idx in range(n_cats, len(axes_flat)):
        axes_flat[idx].set_visible(False)

    fig.suptitle("Price Distribution by Category (Last 30 Days)",
                 fontsize=14, fontweight="bold", color=WOW_PALETTE["text"])
    fig.tight_layout()
    plt.show()
else:
    print("No price data available for distribution plots.")

In [ ]:
# ── Box plot: price ranges by category (log scale) ───────────────────────────

if not df_prices.empty:
    fig, ax = plt.subplots(figsize=(12, 6))
    apply_wow_theme(fig)

    plot_data = df_prices.copy()
    plot_data["log_price"] = np.log10(plot_data["avg_price_gold"].clip(lower=0.01))

    order = (
        plot_data.groupby("category_tag")["avg_price_gold"]
        .median()
        .sort_values(ascending=False)
        .index.tolist()
    )
    palette = {cat: CATEGORY_COLORS.get(cat, WOW_PALETTE["primary"]) for cat in order}

    sns.boxplot(
        data=plot_data, x="category_tag", y="log_price",
        order=order, palette=palette, ax=ax,
        fliersize=2, linewidth=0.8,
        boxprops={"alpha": 0.7},
        medianprops={"color": WOW_PALETTE["accent_red"]},
    )
    ax.set_xlabel("Category")
    ax.set_ylabel("log10(Price in gold)")
    ax.set_title("Price Range by Category", fontweight="bold")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    fig.tight_layout()
    plt.show()
else:
    print("No data for box plot.")

**Key observations:**

- **Gear** and **collection** items span the widest price range (multiple orders of magnitude), making percentage-based metrics (MAPE) essential for fair cross-category comparison.
- **Consumables** and **reagents** cluster in the low-to-mid range but with tight distributions -- these are high-volume, predictable markets.
- **Gems** and **enchants** sit in the mid range with moderate variance -- good forecasting targets.
- Log-scale visualization is critical here; a linear scale would compress the consumable histogram into a single bar.

---
## 3. Temporal Patterns

WoW has strong weekly cycles tied to the **weekly reset** (every Tuesday in US, Wednesday in EU). On reset day:

- **Demand surges** for raid consumables (flasks, potions, food) as guilds prepare for progression.
- **Prices spike** on Tuesday afternoon and decline through the week as supply catches up.
- **Weekend farming** increases supply, often depressing mat prices.

The model captures this via `day_of_week`, `day_of_month`, and `week_of_year` features.

In [ ]:
# ── Day-of-week price patterns ──────────────────────────────────────────────

dow_sql = """
    SELECT CAST(strftime('%w', n.observed_at) AS INTEGER) AS dow,
           ea.category_tag,
           AVG(n.price_gold) AS avg_price
    FROM   market_observations_normalized n
    JOIN   items i ON n.item_id = i.item_id
    JOIN   economic_archetypes ea ON i.archetype_id = ea.archetype_id
    WHERE  n.realm_slug = ?
      AND  n.is_outlier = 0
      AND  date(n.observed_at) >= date('now', '-60 days')
    GROUP BY dow, ea.category_tag
    ORDER BY dow
"""
df_dow = safe_query(DB_PATH, dow_sql, [REALM])

if not df_dow.empty:
    day_names = {0: "Sun", 1: "Mon", 2: "Tue", 3: "Wed", 4: "Thu", 5: "Fri", 6: "Sat"}
    df_dow["day_name"] = df_dow["dow"].map(day_names)

    # Normalize prices within each category to show relative patterns
    def normalize_group(group: pd.DataFrame) -> pd.DataFrame:
        mean_price = group["avg_price"].mean()
        if mean_price > 0:
            group = group.copy()
            group["relative_price"] = group["avg_price"] / mean_price
        else:
            group = group.copy()
            group["relative_price"] = 1.0
        return group

    df_dow = df_dow.groupby("category_tag", group_keys=False).apply(normalize_group)

    fig, ax = plt.subplots(figsize=(12, 6))
    apply_wow_theme(fig)

    # Plot the most interesting categories
    highlight_cats = ["consumable", "mat", "enchant", "gem"]
    day_order = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

    for cat in highlight_cats:
        subset = df_dow[df_dow["category_tag"] == cat].copy()
        if subset.empty:
            continue
        # Reorder by day
        subset["day_order"] = subset["day_name"].map({d: i for i, d in enumerate(day_order)})
        subset = subset.sort_values("day_order")
        color = CATEGORY_COLORS.get(cat, WOW_PALETTE["primary"])
        ax.plot(subset["day_name"], subset["relative_price"],
                marker="o", linewidth=2, label=cat, color=color, markersize=6)

    # Mark reset day
    ax.axvline("Tue", color=WOW_PALETTE["accent_red"], linestyle="--",
               alpha=0.5, linewidth=1.5, label="Weekly Reset (US)")
    ax.axhline(1.0, color=WOW_PALETTE["text_muted"], linestyle=":",
               alpha=0.4, linewidth=1)

    ax.set_xlabel("Day of Week")
    ax.set_ylabel("Relative Price (1.0 = weekly mean)")
    ax.set_title("Day-of-Week Price Patterns by Category", fontweight="bold")
    ax.legend(framealpha=0.8)
    fig.tight_layout()
    plt.show()
else:
    print("No day-of-week data available.")

In [ ]:
# ── Weekly price trends over time ────────────────────────────────────────────

weekly_sql = """
    SELECT strftime('%Y-W%W', n.observed_at) AS year_week,
           ea.category_tag,
           AVG(n.price_gold)   AS avg_price,
           COUNT(*)            AS obs_count
    FROM   market_observations_normalized n
    JOIN   items i ON n.item_id = i.item_id
    JOIN   economic_archetypes ea ON i.archetype_id = ea.archetype_id
    WHERE  n.realm_slug = ?
      AND  n.is_outlier = 0
      AND  date(n.observed_at) >= date('now', '-90 days')
    GROUP BY year_week, ea.category_tag
    ORDER BY year_week
"""
df_weekly = safe_query(DB_PATH, weekly_sql, [REALM])

if not df_weekly.empty:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    apply_wow_theme(fig)

    # Left: price trends
    ax1 = axes[0]
    for cat in sorted(df_weekly["category_tag"].unique()):
        subset = df_weekly[df_weekly["category_tag"] == cat]
        color = CATEGORY_COLORS.get(cat, WOW_PALETTE["primary"])
        ax1.plot(range(len(subset)), subset["avg_price"],
                 label=cat, color=color, linewidth=1.5, alpha=0.8)
    ax1.set_xlabel("Week")
    ax1.set_ylabel("Average Price (gold)")
    ax1.set_title("Weekly Average Price by Category", fontweight="bold")
    ax1.legend(fontsize=7, framealpha=0.8)

    # Right: observation volume over time
    ax2 = axes[1]
    weekly_total = df_weekly.groupby("year_week")["obs_count"].sum().reset_index()
    ax2.bar(range(len(weekly_total)), weekly_total["obs_count"],
            color=WOW_PALETTE["primary"], alpha=0.7,
            edgecolor=WOW_PALETTE["grid"], linewidth=0.5)
    ax2.set_xlabel("Week")
    ax2.set_ylabel("Observation Count")
    ax2.set_title("Weekly Data Volume", fontweight="bold")

    fig.suptitle("90-Day Weekly Trends",
                 fontsize=14, fontweight="bold", color=WOW_PALETTE["text"])
    fig.tight_layout()
    plt.show()
else:
    print("No weekly trend data available.")

**What the temporal features capture:**

- `day_of_week` (ISO 1=Mon to 7=Sun): The Tuesday reset spike is clearly visible in consumable prices. The model learns this cyclical demand pattern.
- `week_of_year`: Seasonal effects -- early in an expansion tier, prices are high and declining; mid-tier they stabilize; pre-patch they spike again.
- `days_since_expansion`: A monotonic feature that captures the overall progression curve. Early expansion = inflation, late expansion = deflation.

---
## 4. Event Impact Visualization

The WoW event system is a first-class concept in the forecaster. Events are seeded from a curated JSON file with:

- **Type**: `patch`, `season_start`, `rtwf`, `hotfix`, `competitive`, etc.
- **Severity**: `minor` through `critical` (controls event_boost magnitude)
- **announced_at**: When the event was publicly known (strict leakage guard -- the model only sees events that were announced before the observation date)

Each event has **category-level impact records** that specify whether prices spike, crash, or stay neutral for each archetype category. These feed directly into the `event_impact_magnitude` and `event_archetype_impact` features.

Let's see which events are in our dataset and how prices moved around them.

In [ ]:
# ── Event timeline ───────────────────────────────────────────────────────────

df_events = fetch_events(DB_PATH, days_back=180, days_ahead=30)

if not df_events.empty:
    print(f"Events in window: {len(df_events)}\n")
    display_cols = ["slug", "display_name", "event_type", "severity", "start_date", "end_date"]
    available_cols = [c for c in display_cols if c in df_events.columns]
    print(df_events[available_cols].to_string(index=False))
else:
    print("No events found. Run `wow-forecaster build-events` to seed the event table.")

In [ ]:
# ── Event impact on consumable prices ────────────────────────────────────────
# Pick a representative archetype and overlay event windows on the price chart.

if not df_archetypes.empty:
    # Find a consumable archetype with data
    consumable_archs = df_archetypes[
        df_archetypes["category_tag"] == "consumable"
    ].head(3)

    for _, arch_row in consumable_archs.iterrows():
        arch_id = arch_row["archetype_id"]
        arch_name = arch_row.get("display_name", f"Archetype {arch_id}")

        df_hist = fetch_historical_prices(DB_PATH, REALM, arch_id, days=120)
        if df_hist.empty or len(df_hist) < 14:
            continue

        fig, ax = plt.subplots(figsize=(14, 5))
        apply_wow_theme(fig)

        dates = pd.to_datetime(df_hist["obs_date"])
        prices = df_hist["avg_price_gold"].astype(float)

        ax.plot(dates, prices, color=WOW_PALETTE["primary"],
                linewidth=1.8, label="Daily Avg Price")
        if "min_price_gold" in df_hist.columns:
            ax.fill_between(dates,
                            df_hist["min_price_gold"].astype(float),
                            df_hist["max_price_gold"].astype(float),
                            alpha=0.12, color=WOW_PALETTE["primary"],
                            label="Daily Range")

        # Overlay events
        if not df_events.empty:
            for _, ev in df_events.iterrows():
                ev_start = pd.to_datetime(ev["start_date"])
                if dates.min() <= ev_start <= dates.max():
                    ax.axvline(ev_start, color=WOW_PALETTE["accent_red"],
                               linestyle="--", alpha=0.6, linewidth=1)
                    ax.text(ev_start, ax.get_ylim()[1] * 0.95,
                            f" {ev['display_name']}", fontsize=7,
                            color=WOW_PALETTE["accent_red"], rotation=90,
                            va="top", ha="left", alpha=0.8)

        ax.set_title(f"{arch_name} -- Price Timeline with Events",
                     fontweight="bold", fontsize=12)
        ax.set_xlabel("Date")
        ax.set_ylabel("Price (gold)")
        ax.legend(framealpha=0.8)
        fig.tight_layout()
        plt.show()
        break  # show just the first one with sufficient data
else:
    print("No archetype data available for event impact visualization.")

In [ ]:
# ── Event category impact heatmap ────────────────────────────────────────────
# Shows expected impact direction and magnitude per category for each event.

impact_sql = """
    SELECT we.display_name AS event_name,
           we.severity,
           eci.category_tag,
           eci.impact_direction,
           eci.expected_magnitude
    FROM   event_category_impacts eci
    JOIN   wow_events we ON eci.event_id = we.event_id
    ORDER  BY we.start_date, eci.category_tag
"""
df_impacts = safe_query(DB_PATH, impact_sql)

if not df_impacts.empty:
    # Pivot to event x category matrix of magnitudes
    direction_sign = df_impacts["impact_direction"].map(
        {"spike": 1, "crash": -1, "neutral": 0, "mixed": 0}
    ).fillna(0)
    df_impacts["signed_magnitude"] = (
        df_impacts["expected_magnitude"].fillna(0) * direction_sign
    )

    pivot = df_impacts.pivot_table(
        index="event_name", columns="category_tag",
        values="signed_magnitude", aggfunc="mean"
    ).fillna(0)

    if not pivot.empty:
        fig, ax = plt.subplots(figsize=(14, max(6, len(pivot) * 0.5)))
        apply_wow_theme(fig)

        sns.heatmap(
            pivot, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            ax=ax, linewidths=0.5, linecolor=WOW_PALETTE["grid"],
            cbar_kws={"label": "Expected Price Impact (+ = spike, - = crash)"},
            annot_kws={"fontsize": 8},
        )
        ax.set_title("Event Impact Matrix: Event x Category", fontweight="bold")
        ax.set_ylabel("")
        fig.tight_layout()
        plt.show()
else:
    print("No event impact data. Run `wow-forecaster build-events` to populate.")

**Event impact patterns:**

- Raid tier releases (`season_start`) cause the strongest consumable demand spikes (flasks, potions, food, enchants, gems all surge).
- RTWF (Race to World First) creates extreme short-lived spikes in consumables as top guilds consume thousands of items.
- Mid-season patches can cause mat price spikes (new recipes require new materials) or crashes (increased drop rates).
- The event impact matrix directly feeds the `event_boost` scoring component, which contributes 15% of the recommendation score.

---
## 5. Cold Start vs Warm Archetypes

A core challenge in the WoW Economy Forecaster is **transfer learning across expansions**. When a new expansion launches (e.g., Midnight), the AH fills with entirely new items -- but many of them are the *functional equivalents* of items from the previous expansion (The War Within).

The forecaster handles this through **archetype-based transfer**:

1. Items are grouped into economic archetypes (e.g., "stat flask", "primary herb", "weapon enchant").
2. Archetypes are mapped across expansions (TWW "stat flask" maps to Midnight "stat flask").
3. For cold-start archetypes with limited history, the model blends its prediction with the source-expansion price anchor, weighted by `transfer_confidence`.

The blending formula: `p_hat = c * p_model + (1 - c) * p_source`

Let's compare the uncertainty profiles of cold-start vs warm archetypes.

In [ ]:
# ── Cold-start vs warm: CI width comparison ─────────────────────────────────

from wow_forecaster.viz.charts.transfer_chart import (
    plot_ci_width_by_category,
    plot_cold_start_blend_diagram,
)

df_forecasts = fetch_forecast_data(DB_PATH, REALM)

if not df_forecasts.empty:
    # Enrich with category_tag from archetypes table
    if not df_archetypes.empty and "archetype_id" in df_forecasts.columns:
        df_forecasts = df_forecasts.merge(
            df_archetypes[["archetype_id", "category_tag"]],
            on="archetype_id", how="left",
        )

    fig = plot_ci_width_by_category(df_forecasts, split_cold_start=True)
    plt.show()
else:
    print("No forecast data. Run `wow-forecaster run-daily-forecast` first.")

In [ ]:
# ── Cold-start blending diagram ──────────────────────────────────────────────
# This is a conceptual diagram -- it does not require live data.

fig = plot_cold_start_blend_diagram()
plt.show()

In [ ]:
# ── Transfer confidence distribution ─────────────────────────────────────────

if not df_archetypes.empty and "transfer_confidence" in df_archetypes.columns:
    tc = df_archetypes["transfer_confidence"].dropna()
    if not tc.empty:
        fig, ax = plt.subplots(figsize=(8, 4))
        apply_wow_theme(fig)

        ax.hist(tc, bins=20, color=WOW_PALETTE["accent_blue"], alpha=0.75,
                edgecolor=WOW_PALETTE["grid"], linewidth=0.5)
        ax.axvline(tc.median(), color=WOW_PALETTE["accent_red"],
                   linestyle="--", linewidth=1.5,
                   label=f"Median: {tc.median():.2f}")
        ax.set_xlabel("Transfer Confidence Score")
        ax.set_ylabel("Count")
        ax.set_title("Transfer Confidence Distribution Across Archetypes",
                     fontweight="bold")
        ax.legend(framealpha=0.8)
        fig.tight_layout()
        plt.show()
    else:
        print("No transfer confidence values found.")
else:
    print("No transfer confidence data in archetypes table.")

**Key takeaways from EDA:**

1. **Price distributions vary by orders of magnitude** across categories, requiring log-scale features and MAPE for fair evaluation.
2. **Strong weekly seasonality** exists, particularly for consumables around the Tuesday reset.
3. **Events create measurable, category-specific price shocks** that are captured in the feature engineering pipeline.
4. **Cold-start archetypes** have wider confidence intervals (appropriately reflecting uncertainty), and the blending mechanism anchors predictions to historical analogues.

These observations directly motivate the 48-feature design explored in Notebook 02.